In [3]:
from logging import setLogRecordFactory
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
import time

In [7]:
class Parser:
    def __init__(self, headers = None):
        if headers is None:
            headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
        self.headers = headers
        self.session = requests.Session()

    def parse_page(self, url):
        a = self.session.get(url, headers=self.headers)
        a.raise_for_status()
        b = BeautifulSoup(a.text, 'html.parser')
        return b

    def stop_parsing(self):
        self.session.close()
    def new_parsing(self):
        self.session = requests.Session()


In [29]:
desserts = Parser()
names_1 = []
prices_1 = []
d_p = desserts.parse_page('https://vkusvill.ru/goods/sladosti-i-deserty/sladosti-do-200-kkal/?PAGEN_1=2')

def parsing_vkusvill(x):
    for i in x.find_all('div', class_='ProductCard__content'):
        tekst = i.find('span', itemprop="name")
        names_1.append(tekst.get_text(strip=True))
        prise = i.find('span', class_='Price subtitle _desktop-md _mobile-sm Price--gray Price--label')
        prices_1.append(prise.get_text(strip=True))
parsing_vkusvill(d_p)

for k in d_p.find_all('a', class_="VV_Pager__Item"):
    q = 'https://vkusvill.ru' + k['href']
    r = desserts.parse_page(q)
    parsing_vkusvill(r)
table_d = pd.DataFrame()
table_d['Название'] = names_1
table_d['Цена'] = prices_1
print(table_d)

                                              Название       Цена
0    Батончик из белевской яблочной пастилы с черно...   79руб/шт
1                   Желе фруктовые и ягодные "Слуперы"  125руб/шт
2                                Зефир воздушный, 15 г   46руб/шт
3    Желе "Шарики" клубника, яблоко-виноград, апель...  276руб/шт
4              Шоколад горячий "Бомбочка" с маршмеллоу  170руб/шт
..                                                 ...        ...
118             Батончик протеиновый с малиной и кешью  187руб/шт
119        Драже Воздушная пшеница в молочном шоколаде   89руб/шт
120    Шоколадная фигурка из молочного шок. с печеньем   82руб/шт
121                    Шоколад без сахара на изомальте  123руб/шт
122               Карамель леденцовая "Ассорти вкусов"   90руб/шт

[123 rows x 2 columns]


In [30]:
table_d['Цена'] = table_d['Цена'].str.replace('руб/шт', '')
table_d['Цена'] = table_d['Цена'].astype(int)
print(table_d)
desserts.stop_parsing()

                                              Название  Цена
0    Батончик из белевской яблочной пастилы с черно...    79
1                   Желе фруктовые и ягодные "Слуперы"   125
2                                Зефир воздушный, 15 г    46
3    Желе "Шарики" клубника, яблоко-виноград, апель...   276
4              Шоколад горячий "Бомбочка" с маршмеллоу   170
..                                                 ...   ...
118             Батончик протеиновый с малиной и кешью   187
119        Драже Воздушная пшеница в молочном шоколаде    89
120    Шоколадная фигурка из молочного шок. с печеньем    82
121                    Шоколад без сахара на изомальте   123
122               Карамель леденцовая "Ассорти вкусов"    90

[123 rows x 2 columns]


In [50]:
driver = webdriver.Chrome()
driver.get('https://nosugarbakery.ru/menu#!/tab/916281403-1')

wait = WebDriverWait(driver, 10)
category_buttons = wait.until(EC.presence_of_all_elements_located(
            (By.CSS_SELECTOR, ".t395__title.t-name.t-name_xs")
        ))


all_dishes = []
for idx, btn in enumerate(category_buttons):
    driver.execute_script("arguments[0].click();", btn)
    time.sleep(3)
    while True:
        try:
                load_more = driver.find_element(By.CSS_SELECTOR, ".t-btn.t-btnflex.t-btnflex_type_button.t-btnflex_xs.js-store-load-more-btn.t-store__load-more-btn")
                load_more.click()
                time.sleep(5)
                break

        except:
                break

    product_cards = driver.find_elements(By.CSS_SELECTOR, ".t-store__card")

    for card in product_cards:
                try:
                    name_elem = card.find_element(By.CLASS_NAME, "t-store__card__title")
                    name = name_elem.text.strip()

                    price_elem = card.find_element(By.CLASS_NAME, "t-store__card__price-value")
                    price = price_elem.text.strip().replace("р.", "").replace(" ", "")

                    if name and price:
                        all_dishes.append({
                            "Название": name,
                            "Цена": price
                        })
                except Exception:
                    continue



table_2 = pd.DataFrame(all_dishes)
print(table_2)


driver.quit()


                                             Название  Цена
0                                     Шоколадный кекс   410
1                                      Пирог с вишней   620
2                                    Баскский чизкейк   790
3                                         Груша-банан   750
4                           Печенье шоколадный брауни   350
5                Печенье овсяное с шоколадной крошкой   350
6                                        Десерт Кокос   730
7                                       Эклер ягодный   790
8                                 Миндальный круассан   750
9                                Клубника со сливками   770
10                                       Десерт Банан   770
11                                        Лесной орех   740
12                     Шоколадный с черной смородиной   790
13                                           Тирамису   770
14                                     Земляника-личи   790
15                                 Трюфе

In [51]:
table_2 = table_2[table_2['Название'] != 'Футболка']
table_2['Цена'] = table_2['Цена'].astype(int)
print(table_2)

                                             Название  Цена
0                                     Шоколадный кекс   410
1                                      Пирог с вишней   620
2                                    Баскский чизкейк   790
3                                         Груша-банан   750
4                           Печенье шоколадный брауни   350
5                Печенье овсяное с шоколадной крошкой   350
6                                        Десерт Кокос   730
7                                       Эклер ягодный   790
8                                 Миндальный круассан   750
9                                Клубника со сливками   770
10                                       Десерт Банан   770
11                                        Лесной орех   740
12                     Шоколадный с черной смородиной   790
13                                           Тирамису   770
14                                     Земляника-личи   790
15                                 Трюфе

In [114]:
table_2.reset_index(drop=True, inplace=True)
print(table_2)

                                             Название  Цена
0                                     Шоколадный кекс   410
1                                      Пирог с вишней   620
2                                    Баскский чизкейк   790
3                                         Груша-банан   750
4                           Печенье шоколадный брауни   350
5                Печенье овсяное с шоколадной крошкой   350
6                                        Десерт Кокос   730
7                                       Эклер ягодный   790
8                                 Миндальный круассан   750
9                                Клубника со сливками   770
10                                       Десерт Банан   770
11                                        Лесной орех   740
12                     Шоколадный с черной смородиной   790
13                                           Тирамису   770
14                                     Земляника-личи   790
15                                 Трюфе

In [110]:
driver = webdriver.Chrome()
driver.get('https://ti.pitadobraw.ru/')
product_cards_2 = driver.find_elements(By.CSS_SELECTOR, ".t-store__card")
all_dishes_2 = []


current_scroll = driver.execute_script("return window.pageYOffset")
max_scroll = driver.execute_script("return document.body.scrollHeight - window.innerHeight")


while current_scroll < max_scroll - 999:

    product_cards_2 = driver.find_elements(By.CSS_SELECTOR, ".t-store__card")

    for card in product_cards_2:
        try:
            name_elem = card.find_element(By.CSS_SELECTOR, ".js-product-name")
            name = name_elem.text.strip()

            price_elem = card.find_element(By.CSS_SELECTOR, ".js-product-price")
            price = price_elem.text.strip().replace(" ", "").replace("р.", "")

            if name and price:
                all_dishes_2.append({
                    "Название": name,
                    "Цена": price
                })
            if name == 'Соус Цезарь':
                break
        except Exception:
            continue
    current_scroll = driver.execute_script("return window.pageYOffset")
    try:
        driver.execute_script("window.scrollBy(0, 1000);")
    except:
        break


while current_scroll < max_scroll:  
    product_cards_2 = driver.find_elements(By.CSS_SELECTOR, ".t-store__card")

    for card in product_cards_2:
        try:
            name_elem = card.find_element(By.CSS_SELECTOR, ".js-product-name")
            name = name_elem.text.strip()

            price_elem = card.find_element(By.CSS_SELECTOR, ".js-product-price")
            price = price_elem.text.strip().replace(" ", "").replace("р.", "")

            if name and price:
                all_dishes_2.append({
                    "Название": name,
                    "Цена": price
                })
        except Exception:
            continue
    current_scroll = driver.execute_script("return window.pageYOffset")
    try:
        driver.execute_script("window.scrollBy(0, 1);")
    except:
        break

driver.execute_script("window.scrollTo(0, document.body.scrollHeight - 100);")
product_cards_2 = driver.find_elements(By.CSS_SELECTOR, ".t-store__card")
for card in product_cards_2:
        try:
            name_elem = card.find_element(By.CSS_SELECTOR, ".js-product-name")
            name = name_elem.text.strip()

            price_elem = card.find_element(By.CSS_SELECTOR, ".js-product-price")
            price = price_elem.text.strip().replace(" ", "").replace("р.", "")

            if name and price:
                all_dishes_2.append({
                    "Название": name,
                    "Цена": price
                })
        except Exception:
            continue


driver.execute_script("window.scrollTo(0, document.body.scrollHeight-10);")
product_cards_2 = driver.find_elements(By.CSS_SELECTOR, ".t-store__card")
for card in product_cards_2:
        try:
            name_elem = card.find_element(By.CSS_SELECTOR, ".js-product-name")
            name = name_elem.text.strip()

            price_elem = card.find_element(By.CSS_SELECTOR, ".js-product-price")
            price = price_elem.text.strip().replace(" ", "").replace("р.", "")

            if name and price:
                all_dishes_2.append({
                    "Название": name,
                    "Цена": price
                })
        except Exception:
            continue

table_3 = pd.DataFrame(all_dishes_2)
table_3 = table_3.drop_duplicates()
print(table_3)

driver.quit()

                                            Название  Цена
0      Шоколад протеиновый с миндалем by СИЛА DOBRAW  3300
1       Шоколад протеиновый с пеканом by СИЛА DOBRAW  3900
2     Шоколад протеиновый с фисташкой by СИЛА DOBRAW  4600
3       Шоколад протеиновый кунжутный by СИЛА DOBRAW  3100
4                                           Геркулес   650
...                                              ...   ...
2562                                   Соус Дзадзыки   120
2563                                     Сырный соус   160
2564                                    Терияки соус    90
2565                                    Тонкацу соус    90
2566                                     Соус Цезарь   150

[126 rows x 2 columns]


In [112]:
table_3.reset_index(drop=True, inplace=True)
print(table_3)


                                           Название  Цена
0     Шоколад протеиновый с миндалем by СИЛА DOBRAW  3300
1      Шоколад протеиновый с пеканом by СИЛА DOBRAW  3900
2    Шоколад протеиновый с фисташкой by СИЛА DOBRAW  4600
3      Шоколад протеиновый кунжутный by СИЛА DOBRAW  3100
4                                          Геркулес   650
..                                              ...   ...
121                                   Соус Дзадзыки   120
122                                     Сырный соус   160
123                                    Терияки соус    90
124                                    Тонкацу соус    90
125                                     Соус Цезарь   150

[126 rows x 2 columns]
